# [F26-04] Phase 2: Data Preprocessing Pipeline
**SmolDocling Paper Implementation** | Track C

Execution of the three datatsets has been done in the following order:
1. **Im2LaTeX-230K** — Equation/Formula Recognition
2. **PubTables-1M** — Table Structure Recognition (OTSL)
3. **DocLayNet v1.2** — Full-Page OCR & Layout

Each section downloads data, preprocesses it, validates output, and shows a brief EDA.

---
##Install Dependencies

In [ ]:
!pip install -q datasets Pillow tqdm pandas

In [ ]:

!nvidia-smi

Tue Mar 31 08:14:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Prevent Colab timeout during long runs
import IPython
IPython.display.Javascript('''
function keepAlive() { document.querySelector("colab-toolbar-button#connect").click(); }
setInterval(keepAlive, 60000);
''')

<IPython.core.display.Javascript object>

---
## 1. Im2LaTeX-230K: Equation Recognition

**What this does:**
- Downloads the dataset from Kaggle
- Converts images to RGB at 120 DPI (matching the paper's rendering resolution)
- Normalizes LaTeX (brace-wrapping bare superscripts/subscripts)
- Wraps each formula in SmolDocling's DocTags format
- Saves a CSV mapping image paths to DocTags ground truth

In [ ]:
IMG_DIR = "/content/im2latex_data/PRINTED_TEX_230k/generated_png_images"
FORMULA_FILE = "/content/im2latex_data/PRINTED_TEX_230k/final_png_formulas.txt"
MAP_FILE = "/content/im2latex_data/PRINTED_TEX_230k/corresponding_png_images.txt"

import os
n_images = len([f for f in os.listdir(IMG_DIR) if f.endswith('.png')])
print(f"Images found: {n_images:,}")

Images found: 235,241


In [ ]:
#Install dataset to direct colab
!pip install -q kaggle

# Set up credentials inline
import os
os.environ['KAGGLE_USERNAME'] = 'Nayab Shahbaz'
os.environ['KAGGLE_KEY'] = 'KGAT_42dbdef33435a94f3765fda4a9ca42d8'

# Download directly to Colab (server-to-server, much faster)
!kaggle datasets download -d gregoryeritsyan/im2latex-230k -p /content/im2latex_data/
!unzip -oq /content/im2latex_data/*.zip -d /content/im2latex_data/

Dataset URL: https://www.kaggle.com/datasets/gregoryeritsyan/im2latex-230k
License(s): CC0-1.0
100% 969M/969M [00:07<00:00, 134MB/s]



### 1b. Run preprocessing

In [ ]:
import os
import re
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# ---------- PATHS ----------
IMG_DIR = "/content/im2latex_data/PRINTED_TEX_230k/generated_png_images"
FORMULA_FILE = "/content/im2latex_data/PRINTED_TEX_230k/final_png_formulas.txt"
MAP_FILE = "/content/im2latex_data/PRINTED_TEX_230k/corresponding_png_images.txt"

print(f"Image dir  : {IMG_DIR}")
print(f"Formulas   : {FORMULA_FILE}")
print(f"Mapping    : {MAP_FILE}")

# ---------- NORMALIZATION ----------
def normalize_latex(text):
    """Collapse whitespace, brace-wrap bare superscripts/subscripts."""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\^([a-zA-Z0-9])', r'^{\1}', text)
    text = re.sub(r'_([a-zA-Z0-9])', r'_{\1}', text)
    return text

# ---------- LOAD MAPPING ----------
with open(FORMULA_FILE, 'r') as f:
    formulas = [line.strip() for line in f]
with open(MAP_FILE, 'r') as f:
    image_names = [line.strip() for line in f]

print(f"\nTotal formulas listed : {len(formulas):,}")
print(f"Total image names     : {len(image_names):,}")

# ---------- PROCESS ----------
data = []
missing = 0

for img_name, formula in tqdm(zip(image_names, formulas), total=len(image_names), desc="Processing"):
    img_path = os.path.join(IMG_DIR, img_name)

    if not os.path.exists(img_path):
        missing += 1
        continue

    # Image preprocessing: RGBA → RGB, set 120 DPI
    with Image.open(img_path) as img:
        img = img.convert('RGB')
        img.save(img_path, dpi=(120, 120))

    # Text preprocessing: normalize + wrap in DocTags
    clean = normalize_latex(formula)
    doctag = f'<formula><loc_0><loc_0><loc_500><loc_500>{clean}</formula>'

    data.append({'image_path': img_path, 'doctag': doctag})

# ---------- SAVE ----------
df = pd.DataFrame(data)
df.to_csv('/content/im2latex_ready.csv', index=False)

print(f"\n{'='*50}")
print(f"Images processed : {len(data):,}")
print(f"Images missing   : {missing:,}")
print(f"Success rate     : {len(data)/(len(data)+missing)*100:.1f}%" if (len(data)+missing) > 0 else "N/A")
print(f"CSV saved to     : /content/im2latex_ready.csv")

Image dir  : /content/im2latex_data/PRINTED_TEX_230k/generated_png_images
Formulas   : /content/im2latex_data/PRINTED_TEX_230k/final_png_formulas.txt
Mapping    : /content/im2latex_data/PRINTED_TEX_230k/corresponding_png_images.txt

Total formulas listed : 238,329
Total image names     : 238,329


Processing:   0%|          | 0/238329 [00:00<?, ?it/s]


Images processed : 238,329
Images missing   : 0
Success rate     : 100.0%
CSV saved to     : /content/im2latex_ready.csv


### 1c. Exploratory Data Analysis with validation

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

df = pd.read_csv('/content/im2latex_ready.csv')
print(f"Total rows: {len(df):,}")
print(f"Columns   : {list(df.columns)}")
print()

# --- Tag validation ---
broken_open  = df[~df['doctag'].str.contains('<formula>')].shape[0]
broken_close = df[~df['doctag'].str.contains('</formula>')].shape[0]
print(f"Missing <formula>  tag: {broken_open}")
print(f"Missing </formula> tag: {broken_close}")
print()

# --- DPI + mode check on first 5 images ---
print("Sample image checks:")
for _, row in df.head(5).iterrows():
    with Image.open(row['image_path']) as img:
        dpi = img.info.get('dpi', 'N/A')
        print(f"  {os.path.basename(row['image_path'])}: mode={img.mode}, dpi={dpi}, size={img.size}")
print()

# --- Formula length distribution ---
df['formula_len'] = df['doctag'].str.len()
print(f"DocTag length stats:")
print(df['formula_len'].describe().to_string())
print()

# --- Show a few samples ---
print("Sample DocTags (first 3):")
for i, row in df.head(3).iterrows():
    print(f"  [{i}] {row['doctag'][:120]}...")

Total rows: 238,329
Columns   : ['image_path', 'doctag']

Missing <formula>  tag: 0
Missing </formula> tag: 0

Sample image checks:
  80f1db54ec657ab.png: mode=RGB, dpi=(119.9896, 119.9896), size=(200, 20)
  4c0c01a5fb03248.png: mode=RGB, dpi=(119.9896, 119.9896), size=(169, 39)
  3f55826fd850d77.png: mode=RGB, dpi=(119.9896, 119.9896), size=(138, 19)
  a6a043f87f6ffdc.png: mode=RGB, dpi=(119.9896, 119.9896), size=(142, 20)
  4d0e536147c711b.png: mode=RGB, dpi=(119.9896, 119.9896), size=(175, 18)

DocTag length stats:
count    238329.000000
mean        219.605142
std         112.791026
min          64.000000
25%         147.000000
50%         190.000000
75%         260.000000
max        2228.000000

Sample DocTags (first 3):
  [0] <formula><loc_0><loc_0><loc_500><loc_500>R _ { 1 2 } K _ { 1 } R _ { 2 1 } d K _ { 2 } = d K _ { 2 } R _ { 1 2 } K _ { 1...
  [1] <formula><loc_0><loc_0><loc_500><loc_500>E _ { n } - E _ { m } = \frac { \lambda ^ { \prime } ( n ^ { 2 } y ^ { 2 } - m ...
  [2]

---
## 2. PubTables-1M :Table Structure Recognition

**What this does:**
- Reads XML annotation files (rows, columns, headers, spanning cells)
- Reads companion word files (OCR text with coordinates)
- Builds a cell grid from row/column intersections
- Matches words to cells via center-point containment
- Generates OTSL tokens: `<ched>`, `<fcel>`, `<ecel>`, `<lcel>`, `<ucel>`, `<xcel>`, `<nl>`
- Saves one `.txt` file per table

### 2a. Download the dataset

PubTables-1M is hosted on GitHub/HuggingFace. We'll download a subset for testing.

**If you already have the `data/annotations/` and `data/words/` folders**, skip this cell.

In [ ]:
import os

# Download Structure_Annotations_Test directly from HuggingFace (only ~30 MB)
URL = "https://huggingface.co/datasets/bsmock/pubtables-1m/resolve/main/PubTables-1M-Structure_Annotations_Test.tar.gz"

os.makedirs('/content/pubtables', exist_ok=True)

print("Downloading Structure_Annotations_Test (~30 MB)...")
!wget -q --show-progress -O /content/pubtables/structure_test.tar.gz "{URL}"

print("\nExtracting...")
!tar -xzf /content/pubtables/structure_test.tar.gz -C /content/pubtables/

# Find the XML files
print("\nLooking for XML files...")
!find /content/pubtables -name '*.xml' | head -5

# Count them
xml_count = !find /content/pubtables -name '*.xml' | wc -l
print(f"\nTotal XML files found: {xml_count[0].strip()}")

/content/pubtables/ 100%[===================>]  28.54M  26.2MB/s    in 1.1s    

Extracting...

Looking for XML files...
/content/pubtables/PMC2268688_table_0.xml
/content/pubtables/PMC5808191_table_2.xml
/content/pubtables/PMC5809909_table_0.xml
/content/pubtables/PMC2977537_table_3.xml
/content/pubtables/PMC2933666_table_1.xml

Total XML files found: 93834


###2a.Finding the dir where xmls are and see one xml file:




In [ ]:
# Find the exact directory where XMLs landed
import glob

xml_files = glob.glob('/content/pubtables/**/*.xml', recursive=True)
if xml_files:
    ANNOTATIONS_DIR = os.path.dirname(xml_files[0])
    print(f"Annotations directory: {ANNOTATIONS_DIR}")
    print(f"Total XML files: {len(xml_files):,}")
    print(f"\nFirst 5 files:")
    for f in sorted(xml_files)[:5]:
        print(f"  {os.path.basename(f)}")
else:
    print("ERROR: No XML files found. Check the extraction above.")


    # Show the first XML file for structure
if xml_files:
    sample = sorted(xml_files)[0]
    print(f"File: {os.path.basename(sample)}")
    print("=" * 60)
    with open(sample, 'r') as f:
        content = f.read()
    # Show first 2000 chars
    print(content[:2000])
    if len(content) > 2000:
        print(f"\n... ({len(content)} total characters)")

Annotations directory: /content/pubtables
Total XML files: 93,834

First 5 files:
  PMC1064078_table_0.xml
  PMC1064078_table_2.xml
  PMC1064078_table_3.xml
  PMC1064078_table_4.xml
  PMC1064078_table_5.xml
File: PMC1064078_table_0.xml
<?xml version="1.0" ?>
<annotation>
   <folder/>
   <filename>PMC1064078_table_0.jpg</filename>
   <path>PMC1064078_table_0.jpg</path>
   <source>
      <database>PubTables1M-Structure</database>
   </source>
   <size>
      <width>682</width>
      <height>251</height>
      <depth>3</depth>
   </size>
   <segmented>0</segmented>
   <object>
      <name>table</name>
      <pose>Frontal</pose>
      <truncated>0</truncated>
      <difficult>0</difficult>
      <occluded>0</occluded>
      <bndbox>
         <xmin>36.5556</xmin>
         <ymin>36.6853</ymin>
         <xmax>643.1767</xmax>
         <ymax>211.7458</ymax>
      </bndbox>
   </object>
   <object>
      <name>table spanning cell</name>
      <pose>Frontal</pose>
      <truncated>0</truncated>
 

### 2b. Run preprocessing

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm.auto import tqdm

# ---------- CONFIG ----------
ANNOTATIONS_PATH = Path(ANNOTATIONS_DIR)
OUTPUT_DIR = Path('/content/pubtables/ground_truth_otsl')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES = None  # None = process all, or set e.g. 500 for quick test


# ---------- STEP 1: READ XML ----------
def read_xml(xml_path):
    """
    Reads one XML annotation file.
    Returns: (img_width, img_height), list of element dicts
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find('size')
    img_w = int(size.find('width').text)
    img_h = int(size.find('height').text)

    objects = []
    for obj in root.findall('object'):
        label = obj.find('name').text
        bnd = obj.find('bndbox')
        x1 = float(bnd.find('xmin').text)
        y1 = float(bnd.find('ymin').text)
        x2 = float(bnd.find('xmax').text)
        y2 = float(bnd.find('ymax').text)

        # Spanning cells (merged cells in Excel terms)
        row_span, col_span = 1, 1
        attrs = obj.find('attributes')
        if attrs is not None:
            for attr in attrs.findall('attribute'):
                name = attr.find('name').text
                value = attr.find('value').text
                if name == 'row_span': row_span = int(value)
                elif name == 'column_span': col_span = int(value)

        objects.append({
            'label': label,
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'row_span': row_span, 'col_span': col_span
        })

    return (img_w, img_h), objects


# ---------- STEP 2: BUILD OTSL (STRUCTURE ONLY) ----------
def build_otsl(objects, img_w, img_h):
    """
    Converts XML objects into OTSL string.
    Structure only — no text in any cell.
    """
    # Separate and sort elements
    rows = sorted([o for o in objects if o['label'] == 'table row'],
                  key=lambda o: (o['y1'] + o['y2']) / 2)
    cols = sorted([o for o in objects if o['label'] == 'table column'],
                  key=lambda o: (o['x1'] + o['x2']) / 2)
    headers = [o for o in objects if o['label'] == 'table column header']
    spans = [o for o in objects if o['label'] in (
        'table spanning cell', 'table projected row header')]

    n_rows, n_cols = len(rows), len(cols)
    if n_rows == 0 or n_cols == 0:
        return ''

    # Build cell grid (intersection of each row and column)
    grid = {}
    for ri, row in enumerate(rows):
        for ci, col in enumerate(cols):
            grid[(ri, ci)] = (
                max(row['x1'], col['x1']),
                max(row['y1'], col['y1']),
                min(row['x2'], col['x2']),
                min(row['y2'], col['y2'])
            )

    # Identify header rows
    header_rows = set()
    for hdr in headers:
        for ri, row in enumerate(rows):
            if min(hdr['y2'], row['y2']) - max(hdr['y1'], row['y1']) > 0:
                header_rows.add(ri)

    # Mark spanning (merged) cells
    span_type = {}
    for span in spans:
        best_overlap, anchor = 0, (0, 0)
        for (ri, ci), (cx1, cy1, cx2, cy2) in grid.items():
            ov = (max(0, min(span['x2'], cx2) - max(span['x1'], cx1)) *
                  max(0, min(span['y2'], cy2) - max(span['y1'], cy1)))
            if ov > best_overlap:
                best_overlap, anchor = ov, (ri, ci)

        ar, ac = anchor
        for dr in range(span['row_span']):
            for dc in range(span['col_span']):
                r, c = ar + dr, ac + dc
                if r >= n_rows or c >= n_cols or (dr == 0 and dc == 0):
                    continue
                if dr > 0 and dc > 0: span_type[(r, c)] = 'X'
                elif dc > 0: span_type[(r, c)] = 'L'
                else: span_type[(r, c)] = 'U'

    # Generate OTSL tokens (structure only)
    tokens = []
    for ri in range(n_rows):
        for ci in range(n_cols):
            st = span_type.get((ri, ci))
            if st == 'L': tokens.append('<lcel>')
            elif st == 'U': tokens.append('<ucel>')
            elif st == 'X': tokens.append('<xcel>')
            elif ri in header_rows: tokens.append('<ched></ched>')
            else: tokens.append('<ecel>')
        tokens.append('<nl>')

    return '<otsl>' + ' '.join(tokens) + '</otsl>'


# ---------- MAIN LOOP ----------
all_xml = sorted(ANNOTATIONS_PATH.glob('*.xml'))
if MAX_SAMPLES is not None:
    all_xml = all_xml[:MAX_SAMPLES]

print(f"Processing {len(all_xml):,} XML files...")
print(f"Output: {OUTPUT_DIR}")
print()

success, skipped = 0, 0
skip_reasons = []

for xml_path in tqdm(all_xml, desc='Converting to OTSL'):
    sample_id = xml_path.stem
    output_path = OUTPUT_DIR / f'{sample_id}.txt'

    try:
        (img_w, img_h), objects = read_xml(xml_path)
    except Exception as e:
        skipped += 1
        skip_reasons.append(f'{sample_id}: XML error — {e}')
        continue

    otsl = build_otsl(objects, img_w, img_h)
    if not otsl:
        skipped += 1
        skip_reasons.append(f'{sample_id}: no rows/cols detected')
        continue

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(otsl)
    success += 1

print(f"\n{'='*50}")
print(f"Successfully converted : {success:,}")
print(f"Skipped               : {skipped:,}")
print(f"Output saved to       : {OUTPUT_DIR}")

if skip_reasons:
    print(f"\nSkip reasons (first 5):")
    for r in skip_reasons[:5]:
        print(f"  {r}")

Processing 93,834 XML files...
Output: /content/pubtables/ground_truth_otsl



Converting to OTSL:   0%|          | 0/93834 [00:00<?, ?it/s]


Successfully converted : 93,834
Skipped               : 0
Output saved to       : /content/pubtables/ground_truth_otsl


### 2c. Validate & EDA

In [ ]:
import re
from collections import Counter
from pathlib import Path

output_dir = Path('/content/pubtables/ground_truth_otsl')
otsl_files = sorted(output_dir.glob('*.txt'))

print(f"Total OTSL files: {len(otsl_files):,}")
print()

# --- Tag frequency ---
all_tags = Counter()
row_counts = []
col_counts = []
has_spans = 0
has_headers = 0

for fp in otsl_files:
    content = fp.read_text(encoding='utf-8')
    tags = re.findall(r'<(\w+)>', content)
    all_tags.update(tags)

    n_nl = content.count('<nl>')
    row_counts.append(n_nl)

    first_row = content.split('<nl>')[0]
    n_cells = len(re.findall(r'<(ched|ecel|lcel|ucel|xcel)>', first_row))
    col_counts.append(n_cells)

    if '<lcel>' in content or '<ucel>' in content or '<xcel>' in content:
        has_spans += 1
    if '<ched>' in content:
        has_headers += 1

print("Tag frequency across all files:")
for tag, count in all_tags.most_common():
    print(f"  <{tag}> : {count:,}")
print()

print(f"Tables with headers      : {has_headers:,} / {len(otsl_files):,} ({has_headers/len(otsl_files)*100:.1f}%)")
print(f"Tables with merged cells : {has_spans:,} / {len(otsl_files):,} ({has_spans/len(otsl_files)*100:.1f}%)")
print()

if row_counts:
    print(f"Rows per table  — min: {min(row_counts)}, max: {max(row_counts)}, avg: {sum(row_counts)/len(row_counts):.1f}")
    print(f"Cols per table  — min: {min(col_counts)}, max: {max(col_counts)}, avg: {sum(col_counts)/len(col_counts):.1f}")
    print()


# --- Show 3 sample outputs ---
print("="*60)
print("SAMPLE OTSL OUTPUTS")
print("="*60)

for fp in sorted(output_dir.glob('*.txt'))[:3]:
    content = fp.read_text(encoding='utf-8')
    print(f"\nFile: {fp.name}")
    print("-" * 40)

    # Pretty print: one row per line
    rows = content.split('<nl>')
    for i, row in enumerate(rows):
        row = row.strip()
        if row:
            print(f"  Row {i+1}: {row} <nl>")
    print()

Total OTSL files: 93,834

Tag frequency across all files:
  <ecel> : 6,286,792
  <nl> : 1,255,017
  <ched> : 754,485

Tables with headers      : 86,917 / 93,834 (92.6%)
Tables with merged cells : 0 / 93,834 (0.0%)

Rows per table  — min: 1, max: 74, avg: 13.4
Cols per table  — min: 2, max: 51, avg: 5.5

SAMPLE OTSL OUTPUTS

File: PMC1064078_table_0.txt
----------------------------------------
  Row 1: <ched></ched> <ched></ched> <ched></ched> <ched></ched> <ched></ched> <ched></ched> <nl>
  Row 2: <ched></ched> <ched></ched> <ched></ched> <ched></ched> <ched></ched> <ched></ched> <nl>
  Row 3: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>
  Row 4: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>
  Row 5: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>
  Row 6: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>
  Row 7: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>
  Row 8: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>
  Row 9: <ecel> <ecel> <ecel> <ecel> <ecel> <ecel> <nl>


File: PMC10640

---
## 3. DocLayNet v1.2: Full-Page OCR & Layout

**What this does:**
- Loads DocLayNet v1.2 test split from HuggingFace (streaming)
- Converts COCO bounding boxes([x, y, width, height]) to SmolDocling's 500-grid coordinate system
- Sorts elements in reading order (top-to-bottom, left-to-right)
- Wraps everything in DocTags format with `<doctag>` root
- Saves page images alongside the JSONL ground truth

### 3a. Run preprocessing

In [ ]:
import json
import os
from datasets import load_dataset
from tqdm.auto import tqdm

# ---------- CONFIG ----------
GRID_SIZE    = 500
OUTPUT_JSONL = '/content/doclaynet_ready.jsonl'
IMAGE_DIR    = '/content/doclaynet_images'
os.makedirs(IMAGE_DIR, exist_ok=True)

CATEGORIES = {
    1: 'caption',    2: 'footnote',      3: 'formula',
    4: 'list_item',  5: 'page_footer',   6: 'page_header',
    7: 'picture',    8: 'section_header', 9: 'table',
    10: 'text',      11: 'title'
}

def quantize(value, max_val):
    bin_val = int(round((value / max_val) * (GRID_SIZE - 1)))
    return max(0, min(GRID_SIZE - 1, bin_val))

print('Loading DocLayNet v1.2 test split(5000 pages almost) (streaming)...')
ds = load_dataset('docling-project/DocLayNet-v1.2', split='test', streaming=True)

with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for i, example in enumerate(tqdm(ds, desc='Processing')):

        img = example['image']
        w, h = img.size

        img_path = os.path.join(IMAGE_DIR, f'page_{i:05d}.png')
        img.save(img_path)

        bboxes   = example['bboxes']
        cat_ids  = example['category_id']
        texts    = example['pdf_cells']

        elements = []
        for idx in range(len(bboxes)):
            content = ' '.join([cell['text'] for cell in texts[idx]]).strip()
            elements.append({
                'bbox': bboxes[idx],
                'cat_id': cat_ids[idx],
                'text': content
            })

        elements.sort(key=lambda e: (e['bbox'][1], e['bbox'][0]))

        doctags_str = '<doctag>'
        for el in elements:
            x, y, bw, bh = el['bbox']
            tag = CATEGORIES.get(el['cat_id'], 'text')
            x1 = quantize(x, w)
            y1 = quantize(y, h)
            x2 = quantize(x + bw, w)
            y2 = quantize(y + bh, h)
            doctags_str += f'<{tag}><loc_{x1}><loc_{y1}><loc_{x2}><loc_{y2}>{el["text"]}</{tag}>'
        doctags_str += '</doctag>'

        record = {
            'index': i,
            'image_path': img_path,
            'doc_category': example.get('metadata', {}).get('doc_category', 'unknown'),
            'ground_truth_doctags': doctags_str
        }
        f.write(json.dumps(record) + '\n')

print(f"\n{'='*50}")
print(f"Processed  : {i+1} pages")
print(f"JSONL saved: {OUTPUT_JSONL}")
print(f"Images in  : {IMAGE_DIR}")

Loading DocLayNet v1.2 test split(5000 pages almost) (streaming)...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Processing: 0it [00:00, ?it/s]

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/docling-project/DocLayNet-v1.2/resolve/0daf93102e2efce76c3e11a274a5e0d0969391d3/data/test-00004-of-00006.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/docling-project/DocLayNet-v1.2/resolve/0daf93102e2efce76c3e11a274a5e0d0969391d3/data/test-00005-of-00006.parquet
Retrying in 1s [Retry 1/5].



Processed  : 4999 pages
JSONL saved: /content/doclaynet_ready.jsonl
Images in  : /content/doclaynet_images


### 3b. Validate & EDA

In [ ]:
import json
import re
from collections import Counter

records = []
with open('/content/doclaynet_ready.jsonl', 'r') as f:
    for line in f:
        records.append(json.loads(line))

print(f"Total pages: {len(records)}")
print()

# --- Document category distribution ---
cat_counts = Counter(r['doc_category'] for r in records)
print("Document categories:")
for cat, count in cat_counts.most_common():
    print(f"  {cat}: {count}")
print()

# --- Tag frequency ---
tag_counter = Counter()
for r in records:
    tags = re.findall(r'<(text|caption|footnote|formula|list_item|page_footer|page_header|picture|section_header|table|title)>', r['ground_truth_doctags'])
    tag_counter.update(tags)

print("Element types across all pages:")
for tag, count in tag_counter.most_common():
    pct = count / sum(tag_counter.values()) * 100
    print(f"  <{tag}>: {count} ({pct:.1f}%)")
print()

# --- DocTag validation ---
missing_doctag_open  = sum(1 for r in records if not r['ground_truth_doctags'].startswith('<doctag>'))
missing_doctag_close = sum(1 for r in records if not r['ground_truth_doctags'].endswith('</doctag>'))
print(f"Missing <doctag> wrapper  : {missing_doctag_open}")
print(f"Missing </doctag> wrapper : {missing_doctag_close}")
print()

# --- Elements per page ---
elems_per_page = []
for r in records:
    n = len(re.findall(r'<loc_\d+>', r['ground_truth_doctags'])) // 4
    elems_per_page.append(n)

print(f"Elements per page — min: {min(elems_per_page)}, max: {max(elems_per_page)}, avg: {sum(elems_per_page)/len(elems_per_page):.1f}")
print()

# --- Show first sample ---
print("Sample (page 0, first 300 chars):")
print(f"  {records[0]['ground_truth_doctags'][:300]}...")

Total pages: 4999

Document categories:
  financial_reports: 1755
  scientific_articles: 943
  manuals: 802
  laws_and_regulations: 784
  patents: 442
  government_tenders: 273

Element types across all pages:
  <text>: 29940 (45.0%)
  <list_item>: 10522 (15.8%)
  <section_header>: 8550 (12.9%)
  <page_footer>: 3994 (6.0%)
  <picture>: 3534 (5.3%)
  <page_header>: 3366 (5.1%)
  <table>: 2394 (3.6%)
  <formula>: 1966 (3.0%)
  <caption>: 1543 (2.3%)
  <footnote>: 387 (0.6%)
  <title>: 335 (0.5%)

Missing <doctag> wrapper  : 0
Missing </doctag> wrapper : 0

Elements per page — min: 0, max: 102, avg: 13.3

Sample (page 0, first 300 chars):
  <doctag><picture><loc_212><loc_0><loc_499><loc_287></picture><caption><loc_102><loc_15><loc_187><loc_34>Leigh Taliaferro, M.D. General Surgeon Abilene, Texas</caption><section_header><loc_33><loc_55><loc_174><loc_61>Leigh Taliaferro, M.D., values consistency.</section_header><text><loc_33><loc_65><l...
